# Policy Engine — Dev Log

## Objetivo do módulo

O **Policy Engine** é o motor de regras que traduz obrigações da LGPD (Lei Geral de
Proteção de Dados) aplicadas a sistemas de IA em decisões executáveis. Dado um cenário
de tratamento de dados — quais **categorias de dado** estão envolvidas (Art. 5º), qual
**base legal** ampara o tratamento (Art. 7º/11º) e qual o **contexto operacional**
(envolve menor de idade? decisão totalmente automatizada? transferência internacional?
RIPD já foi feito?) — o motor retorna uma ou mais `PolicyDecision`, cada uma com um
status (`allow`, `allow_with_mitigation`, `deny`, `requires_human_review`), o racional
legal e mitigações concretas sugeridas.

## Papel no pipeline do Governance Copilot

```
PII Detection ──► categorias de dado detectadas ─┐
                                                   ├──► Policy Engine ──► PolicyDecision[]
Input do usuário (base legal, contexto) ──────────┘                         │
                                                                             ▼
                                                              RIPD Engine / Trust Score /
                                                              Explainability / Audit Log
```

O Policy Engine é consumido pelo **Governance Copilot** (orquestrador central) como uma
das entradas do relatório de RIPD (`RIPDReport.policy_decisions`, ver
`shared/schemas.py`). Ele não decide sozinho a "aprovação final" do uso de IA — reporta
todas as políticas aplicáveis para que o orquestrador (e, em última instância, um
humano) tenha o quadro completo de riscos e obrigações antes de liberar o tratamento.


## Decisões de design e trade-offs

**Por que YAML declarativo em vez de código Python para as políticas?**
Políticas de compliance mudam com mais frequência que a lógica de avaliação em si (nova
orientação da ANPD, novo precedente, ajuste de mitigação sugerida). Manter as políticas
em `policies.yaml` — com uma gramática pequena de `trigger` (condições de
aplicabilidade) e `outcomes` ordenados (branching por base legal / sinais de contexto)
— permite editar/adicionar políticas sem tocar em `engine.py`, e permite que um
especialista jurídico revise o arquivo sem ler Python.

**Por que múltiplas políticas podem se aplicar ao mesmo cenário?**
Um cenário real de LGPD raramente é "uma regra só". Dado de saúde sobre um menor de
idade aciona *simultaneamente* a política de dado sensível de saúde (POL-001) e a
política de menor de idade (POL-003) — cada uma com seu próprio racional e mitigação.
Colapsar isso em uma única decisão "vencedora" destruiria informação que o Governance
Copilot (e o RIPD final) precisa. Por isso `evaluate()` retorna `list[PolicyDecision]`,
não uma decisão única — a agregação/priorização entre decisões concorrentes (ex.: o que
prevalece quando há `deny` e `allow_with_mitigation` juntos) é deixada explicitamente
para o `governance_copilot`, que tem visão do RIPD completo.

**Por que `context: dict` livre em vez de um schema Pydantic próprio de contexto?**
Os sinais de contexto relevantes para políticas de IA são heterogêneos e vão crescer
com V2 (novas políticas, novos sinais). Fixar um schema rígido agora forçaria
`shared/schemas.py` a mudar a cada nova política — violando a regra de que mudanças ali
são breaking change para todos os módulos. Um `dict` documentado no cabeçalho do
`policies.yaml` (chaves esperadas, tipos) dá a flexibilidade necessária sem tocar no
contrato compartilhado. O trade-off é perda de checagem estática das chaves de
contexto — mitigado por testes que cobrem cada chave usada.

**Por que "trigger casou mas nenhum outcome casou" não produz decisão nenhuma (e não
uma decisão default silenciosa)?** Regra de qualidade do projeto: nunca inventar
cobertura legal que a política não define de fato. Se uma política está mal configurada
para um cenário, o motor simplesmente não relata decisão daquela política — não força
um resultado genérico. Isso é coberto pelo teste `test_evaluate_with_custom_policies_path`.


In [1]:
from core.policy_engine.engine import evaluate
from shared.schemas import DataCategory, LegalBasis


def show(title, decisions):
    print(f"=== {title} ===")
    if not decisions:
        print("  (nenhuma política aplicável)")
    for d in decisions:
        print(f"  [{d.policy_id}] status={d.status.value} risk={d.risk_level.value}")
        print(f"      rationale: {d.rationale}")
        if d.mitigations:
            print(f"      mitigações: {d.mitigations}")
    print()


# Exemplo 1: dado de saúde, com consentimento explícito e RIPD concluído
decisions_1 = evaluate(
    data_categories=[DataCategory.SENSITIVE],
    legal_basis=LegalBasis.CONSENT,
    context={"data_subtype": "health", "ripd_conducted": True, "purpose_specified": True},
)
show("Exemplo 1: dado de saúde, consentimento + RIPD OK", decisions_1)

# Exemplo 2: dado de menor de idade sem consentimento do responsável + dado de saúde sem RIPD
# (mostra múltiplas políticas disparando para o mesmo cenário)
decisions_2 = evaluate(
    data_categories=[DataCategory.SENSITIVE],
    legal_basis=LegalBasis.CONSENT,
    context={"data_subtype": "health", "ripd_conducted": False, "involves_minor": True},
)
show("Exemplo 2: dado de saúde sem RIPD + menor sem consentimento do responsável", decisions_2)

# Exemplo 3: dado pessoal comum, uso corriqueiro (baseline allow)
decisions_3 = evaluate(
    data_categories=[DataCategory.PERSONAL],
    legal_basis=LegalBasis.CONTRACT_EXECUTION,
    context={"purpose_specified": True},
)
show("Exemplo 3: dado pessoal comum, base legal e finalidade definidas", decisions_3)

# Exemplo 4: dado sem nenhuma política aplicável (caso de borda)
decisions_4 = evaluate(
    data_categories=[DataCategory.NOT_PERSONAL],
    legal_basis=LegalBasis.LEGITIMATE_INTEREST,
    context=None,
)
show("Exemplo 4: dado not_personal, nenhum sinal especial (caso de borda)", decisions_4)


=== Exemplo 1: dado de saúde, consentimento + RIPD OK ===
  [POL-001] status=allow_with_mitigation risk=high
      rationale: Dado de saúde processado com consentimento explícito e RIPD concluído. Uso permitido sob controles reforçados (Art. 11, I combinado com Art. 38 LGPD).
      mitigações: ['Restringir acesso ao dado de saúde por controle de papéis (RBAC)', 'Manter o RIPD atualizado a cada mudança relevante de finalidade', 'Auditoria trimestral do uso do dado de saúde pelo sistema de IA']

=== Exemplo 2: dado de saúde sem RIPD + menor sem consentimento do responsável ===
  [POL-001] status=requires_human_review risk=critical
      rationale: Dado de saúde sem consentimento explícito registrado e/ou sem RIPD concluído. Art. 11, I e Art. 38 da LGPD exigem consentimento específico e destacado e RIPD para tratamento de dado sensível em larga escala — revisão humana obrigatória antes de prosseguir.
      mitigações: ['Obter consentimento explícito e destacado do titular (Art. 11, I)', '

In [2]:
import subprocess
import sys

REPO_ROOT = r"g:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\LGPD e IA (Terminar)"
PY = sys.executable  # mesmo interpretador do venv usado para rodar este notebook

result = subprocess.run(
    [PY, "-m", "pytest", "core/policy_engine/tests", "-v"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
print("RETURN CODE:", result.returncode)


============================= test session starts =============================
platform win32 -- Python 3.10.8, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\Yuri_\.venvs\athenagov-ai\Scripts\python.exe
cachedir: .pytest_cache
rootdir: g:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\LGPD e IA (Terminar)
plugins: anyio-4.14.2, cov-7.1.0
collecting ... collected 25 items

core/policy_engine/tests/test_engine.py::test_health_data_with_consent_and_ripd_allows_with_mitigation PASSED [  4%]
core/policy_engine/tests/test_engine.py::test_health_data_without_consent_or_ripd_requires_human_review PASSED [  8%]
core/policy_engine/tests/test_engine.py::test_biometric_automated_decision_without_human_review_denies PASSED [ 12%]
core/policy_engine/tests/test_engine.py::test_biometric_with_human_review_allows_with_mitigation PASSED [ 16%]
core/policy_engine/tests/test_engine.py::test_biometric_without_automation_or_review_signal_requi

## Handoff Summary

### Capacidades entregues

- Motor de política declarativo (`core/policy_engine/engine.py` + `policies.yaml`)
  cobrindo 9 políticas reais de LGPD para IA (saúde, biométrico, menor de idade,
  transferência internacional, dado anonimizado, decisão automatizada com efeito
  jurídico/Art. 20, finalidade não especificada, base legal não determinada, e uma
  política de linha de base para dado pessoal comum).
- Cada política tem `trigger` (quando é relevante) e `outcomes` ordenados (branching
  real por base legal e sinais de contexto — não é só "match categoria → resultado
  fixo").
- Suporte nativo a múltiplas decisões simultâneas para o mesmo cenário.
- Suíte de testes pytest cobrindo as 9 políticas, os 4 valores de
  `PolicyDecisionStatus`, o caso de borda "nenhuma política aplicável", decisões
  simultâneas, e robustez de contrato.

### Assinatura pública exata

```python
def evaluate(
    data_categories: list[DataCategory],
    legal_basis: LegalBasis,
    context: dict[str, Any] | None = None,
    policies_path: Path | str | None = None,
) -> list[PolicyDecision]:
    ...
```

Importação: `from core.policy_engine.engine import evaluate` (ou
`from core.policy_engine import evaluate`, reexportado em `__init__.py`).

### Limitações conhecidas

- O motor **não prioriza/agrega** decisões concorrentes (ex.: quando uma política diz
  `deny` e outra diz `allow_with_mitigation` para o mesmo cenário) — ele apenas relata
  todas. A agregação final é responsabilidade do `governance_copilot`.
- `context` é um `dict` livre, sem validação de schema — chaves incorretas são
  silenciosamente ignoradas (o `trigger`/`when` correspondente simplesmente não casa).
  Isso é uma escolha deliberada (ver seção de trade-offs acima), não um bug, mas é uma
  superfície de erro silencioso a se ter em mente ao integrar.
- As 9 políticas cobrem os cenários explicitamente pedidos no escopo do módulo; não é
  uma cobertura exaustiva de toda a LGPD (isso nunca foi o objetivo do V1).

### O que fica para V2

- TODO: motor de priorização/conflito entre `PolicyDecision` concorrentes (hoje isso é
  implícito, delegado ao consumidor).
- TODO: schema de contexto tipado (Pydantic) quando o vocabulário de sinais estabilizar
  o suficiente para justificar a rigidez adicional.
- TODO: versionamento de políticas individuais (hoje `policies.yaml` é um arquivo
  único, sem histórico de qual política estava vigente em que data — relevante para
  auditoria retroativa).
- Itens mais amplos (Constitutional AI, Regulatory Auto-Update) já estão mapeados no
  `ROADMAP.md` raiz, seção V2/V3, e não são deste módulo.

### Estado dos testes nesta execução

Execução real (não simulada) via
`.venv/Scripts/python.exe -m pytest core/policy_engine/tests -v`, capturada na célula
acima. Return code: 0.
